# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local **agentic workflow** for English-to-Chinese cross-lingual dialogue summarization.

The proposed pipeline is:

```text
English Dialogue
→ Agent 1: English Summarization
→ Agent 2: Chinese Translation
→ Final Chinese Summary
```

The agents are controlled in this notebook. Ollama only serves the local models.


## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [131]:
# Cell 1: Install required Python packages
# Run this only once if the packages are not installed.

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [132]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [ ]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

MAIN_MODEL = "qwen3.5:9b"
TRANSLATION_MODEL = "translategemma:12b"

# If translategemma:12b is too slow, use:
# TRANSLATION_MODEL = "translategemma:4b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_OUTPUT_PATH = OUTPUT_DIR / "two_agent_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "two_agent_final_summaries.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "two_agent_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_outputs.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_final_summaries.csv


In [134]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])
        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")
        return True
    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))
        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [135]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 600,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [136]:
# Cell 5: Prompt templates

SUMMARIZATION_PROMPT = """You are an English dialogue summarization agent.

Your task is to summarize the given English dialogue into a concise English summary.

Requirements:
- Preserve the main intent, key events, decisions, and important entities.
- Do not add information that is not supported by the dialogue.
- Do not include minor details unless they are necessary for understanding the main point.
- Keep the summary concise.
- Use clear and natural English.
- Output only the English summary.

Input dialogue:
{dialogue}

English summary:
"""

TRANSLATION_PROMPT = """You are a professional English-to-Simplified-Chinese translation agent.

Your task is to translate the given English summary into natural Simplified Chinese.

Requirements:
- Preserve the original meaning accurately.
- Preserve names, numbers, dates, and important entities.
- Use Chinese transliterations for common English names.
- Do not include the original English names in parentheses unless necessary.
- Do not add new information.
- Do not remove important information.
- Use fluent and natural Simplified Chinese.
- Output only the Chinese translation.

English summary:
{english_summary}

Chinese summary:
"""

In [137]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)
    return {str(record["id"]) for record in records if "id" in record}

In [138]:
# Cell 7: Agent functions

def summarization_agent(dialogue: str) -> str:
    prompt = SUMMARIZATION_PROMPT.format(dialogue=dialogue)

    return call_ollama(
        model=MAIN_MODEL,
        prompt=prompt,
        temperature=0.2,
    )


def translation_agent(english_summary: str) -> str:
    prompt = TRANSLATION_PROMPT.format(english_summary=english_summary)

    return call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [139]:
# Cell 8: Two-agent pipeline

def run_two_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    english_summary = summarization_agent(dialogue)
    chinese_summary = translation_agent(english_summary)

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "agent1_english_summary": english_summary,
        "agent2_chinese_summary": chinese_summary,
        "final_summary": chinese_summary,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [140]:
# Cell 9: Load top 5 examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [141]:
# Cell 10: Run the two-agent pipeline for the first example from test.json

result = run_two_agent_pipeline(test_data[0])
result

{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_english_summary': "Hannah asks Amanda for Betty's phone number, but Amanda cannot find it and suggests contacting Larry, who last spoke with Betty at the park. Although Hannah is hesitant to reach out to Larry directly, Amanda encourages her to do so, and Hannah reluctantly agrees before ending the conversation.",
 'agent2_chinese_summary': '汉娜向曼达询问贝蒂的电

In [142]:
# Cell 11: Print two-agent pipeline result clearly

def print_two_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: English Summary ===")
    print(result["agent1_english_summary"])
    print()

    print("=== Agent 2: Chinese Summary ===")
    print(result["agent2_chinese_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])


print_two_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: English Summary ===
Hannah asks Amanda for Betty's phone number, but Amanda cannot find it and suggests contacting Larry, who last spoke with Betty at the park. Although Hannah is hesitant to reach out to Larry directly, Amanda encourages her to do so, and Hannah reluctantly agrees before ending the conversation.

=== Agent 2: Chinese Summary ===
汉娜向曼达询问贝蒂的电话号码，但曼达找不到，建议她联系拉里，因为拉里是上次和贝蒂在公园里见过的人。虽然汉娜不太愿意直接联系拉里，但曼达鼓励她这样做，汉娜最终勉强同意，然后结束了对话。

=== Reference English Summary ===
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact

## 4. Save Results

This saves all intermediate outputs and the final output.


In [143]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_outputs.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_final_summaries.csv


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [144]:
# Cell 13: Batch inference with two-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running two-agent pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_two_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running two-agent pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_outputs.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [145]:
# Cell 14: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "agent1_english_summary": record.get("agent1_english_summary", ""),
        "agent2_chinese_summary": record.get("agent2_chinese_summary", ""),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_final_summaries.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_english_summary,agent2_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",汉娜向曼达询问贝蒂的电话号码，但曼达找不到，建议她联系拉里。拉里在他们上次去公园时曾给贝蒂打...,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,"Hannah asks Amanda for Betty's phone number, b...",汉娜向曼达询问贝蒂的电话号码，但曼达找不到，建议她联系拉里。拉里在他们上次去公园时曾给贝蒂打...
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,埃里克和罗布讨论了一段关于一位名为MACHINE的脱口秀喜剧演员的视频，他们称赞了他的幽默以...,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,Eric and Rob discuss a video of a stand-up com...,埃里克和罗布讨论了一段关于一位名为MACHINE的脱口秀喜剧演员的视频，他们称赞了他的幽默以...
2,test_00003,"Lenny: Babe, can you help me with something?\r...",莱尼向鲍勃寻求帮助，想在三件裤子中选择哪一件。鲍勃更喜欢第一件，但莱尼担心自己已经有四件黑色...,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,Lenny asks Bob for help choosing between three...,莱尼向鲍勃寻求帮助，想在三件裤子中选择哪一件。鲍勃更喜欢第一件，但莱尼担心自己已经有四件黑色...
3,test_00004,"Will: hey babe, what do you want for dinner to...",威廉问艾玛晚上想不想一起吃晚饭，但艾玛拒绝了，因为她现在不饿，而且不想太在意这些事情。威廉提...,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,"Will asks Emma about dinner, but she declines ...",威廉问艾玛晚上想不想一起吃晚饭，但艾玛拒绝了，因为她现在不饿，而且不想太在意这些事情。威廉提...
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",简确认她目前在华沙，并与奥利约定周五晚上6点见面喝茶，因为她白天课程太忙，没时间吃午饭。他们...,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,Jane confirms she is in Warsaw and arranges to...,简确认她目前在华沙，并与奥利约定周五晚上6点见面喝茶，因为她白天课程太忙，没时间吃午饭。他们...
